In [0]:
# Persistent paths on Unity Catalog Volumes (survives serverless restarts)
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"

CHROMA_DB_CANDIDATES = [
    "/Volumes/workspace/legal_data/chroma_db/",
    "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test",
    "/Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test",
]

COLLECTION_NAME = "legal_knowledge"
PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"

BATCH_SIZE = 128
FORCE_REBUILD_CHROMA = False
ENABLE_CHROMA_PERSISTENCE = True
CHROMA_RESET_ON_TENANT_ERROR = True



In [0]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import json
import shutil
import traceback
from datetime import datetime, timezone

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F


CELL_DIAGNOSTICS = []


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def report_cell(cell_name: str, ok: bool, details: str):
    status = "OK" if ok else "FAIL"
    entry = {
        "cell": cell_name,
        "status": status,
        "details": details,
        "ts": datetime.now(timezone.utc).isoformat(),
    }
    CELL_DIAGNOSTICS.append(entry)
    print(f"[DIAG] {cell_name} -> {status}: {details}")


def print_diagnostics_summary():
    print("\n=== CELL DIAGNOSTICS SUMMARY ===")
    for d in CELL_DIAGNOSTICS:
        print(f"{d['cell']}: {d['status']} | {d['details']}")



In [0]:
try:
    gold_df = spark.read.format("delta").load(GOLD_PATH)

    required_cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name"]
    missing_cols = [c for c in required_cols if c not in gold_df.columns]
    if missing_cols:
        raise ValueError(f"Gold table is missing required columns: {missing_cols}")

    gold_df = (
        gold_df.select(*required_cols)
        .dropna(subset=["chunk_id", "chunk_text"])
        .dropDuplicates(["chunk_id"])
    )

    gold_count = gold_df.count()
    if gold_count == 0:
        raise ValueError("Gold dataset is empty after filtering. Cannot build embeddings.")

    log(f"Gold chunks ready: {gold_count}")
    gold_df.show(10, truncate=120)
    report_cell("Cell 3 - Load Gold", True, f"rows={gold_count}")

except Exception as e:
    report_cell("Cell 3 - Load Gold", False, str(e))
    raise



[12:10:45] Gold chunks ready: 5194
+------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------+--------------+------------+----------------------------+
|                            chunk_id|                                                                                                              chunk_text|                act_name|section_number|    category|                   file_name|
+------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------+--------------+------------+----------------------------+
|000ae362-e61b-404f-82be-13a3a3542471|wing that A did not intend to harm the reputation of B. (f) A is sued by B for fraudulently representing to B that C ...|Indian Evidence Act 1872|          NULL|criminal_law|Indian Evidence Act_1872.pd

In [0]:
embedding_model = None
loaded_model_name = None

for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
    try:
        log(f"Loading embedding model: {model_name}")
        embedding_model = SentenceTransformer(model_name)
        _ = embedding_model.encode(["health check"], show_progress_bar=False)
        loaded_model_name = model_name
        log(f"Embedding model loaded: {model_name}")
        break
    except Exception as e:
        log(f"Failed to load {model_name}: {e}")

if embedding_model is None:
    report_cell("Cell 4 - Load Embedding Model", False, "No embedding model could be loaded")
    raise RuntimeError("Could not load any embedding model. Check internet/HuggingFace access.")

report_cell("Cell 4 - Load Embedding Model", True, f"model={loaded_model_name}")



[12:10:46] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[12:10:47] Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
[DIAG] Cell 4 - Load Embedding Model -> OK: model=sentence-transformers/all-MiniLM-L6-v2


In [0]:
os.makedirs("/Volumes/workspace/legal_data/vector_db_test", exist_ok=True)
os.makedirs("/Volumes/workspace/legal_data/chroma_db", exist_ok=True)

CHROMA_DB_PATH = None
client = None
collection = None
init_errors = []


def expand_fs_paths(path):
    paths = [path]
    if path.startswith("/Volumes/"):
        paths.append("/dbfs" + path)
    # Deduplicate preserve order
    out = []
    seen = set()
    for p in paths:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def reset_chroma_path(path):
    # Remove only known chroma sqlite artifacts
    names = [
        "chroma.sqlite3",
        "chroma.sqlite3-wal",
        "chroma.sqlite3-shm",
        "chroma.sqlite3-journal",
    ]
    for n in names:
        fp = os.path.join(path, n)
        if os.path.exists(fp):
            os.remove(fp)

    # Remove known chroma folders when present
    for d in ["index", "chroma"]:
        dp = os.path.join(path, d)
        if os.path.isdir(dp):
            shutil.rmtree(dp, ignore_errors=True)


def try_init_chroma(path):
    local_client = chromadb.PersistentClient(
        path=path,
        settings=Settings(anonymized_telemetry=False, allow_reset=True),
    )

    if FORCE_REBUILD_CHROMA:
        try:
            local_client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

    local_collection = local_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    # dimension compatibility check
    sample = embedding_model.encode(["dimension probe"], show_progress_bar=False)[0]
    expected_dim = len(sample)

    if local_collection.count() > 0:
        probe = local_collection.get(limit=1, include=["embeddings"])
        existing_embs = probe.get("embeddings") or []
        if existing_embs and len(existing_embs[0]) != expected_dim:
            local_client.delete_collection(COLLECTION_NAME)
            local_collection = local_client.get_or_create_collection(
                name=COLLECTION_NAME,
                metadata={"hnsw:space": "cosine"}
            )

    return local_client, local_collection


if ENABLE_CHROMA_PERSISTENCE:
    for candidate in CHROMA_DB_CANDIDATES:
        candidate_ok = False
        for fs_path in expand_fs_paths(candidate):
            try:
                os.makedirs(fs_path, exist_ok=True)

                # simple write test
                probe_file = os.path.join(fs_path, ".write_probe")
                with open(probe_file, "w", encoding="utf-8") as f:
                    f.write("ok")
                os.remove(probe_file)

                client, collection = try_init_chroma(fs_path)
                CHROMA_DB_PATH = fs_path
                candidate_ok = True
                break

            except Exception as e:
                err = str(e)

                # Recovery for known tenant metadata corruption
                if CHROMA_RESET_ON_TENANT_ERROR and "default_tenant" in err.lower():
                    try:
                        log(f"Tenant error at {fs_path}; resetting local chroma metadata and retrying")
                        reset_chroma_path(fs_path)
                        client, collection = try_init_chroma(fs_path)
                        CHROMA_DB_PATH = fs_path
                        candidate_ok = True
                        break
                    except Exception as e2:
                        init_errors.append(f"{fs_path}: {e2}")
                else:
                    init_errors.append(f"{fs_path}: {e}")

        if candidate_ok:
            break

if collection is None:
    report_cell(
        "Cell 5 - Init Chroma",
        False,
        "Chroma init failed. Delta embeddings will still be persisted. " + " | ".join(init_errors[:3])
    )
else:
    report_cell("Cell 5 - Init Chroma", True, f"path={CHROMA_DB_PATH}, existing_vectors={collection.count()}")

manifest_path = "/Volumes/workspace/legal_data/vector_db_test/embedding_runtime_manifest.json"
manifest = {
    "embedding_delta_path": EMBEDDING_DELTA_PATH,
    "chroma_path": CHROMA_DB_PATH,
    "collection_name": COLLECTION_NAME,
    "embedding_model": loaded_model_name,
    "chroma_enabled": ENABLE_CHROMA_PERSISTENCE,
    "chroma_init_errors": init_errors,
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log(f"Wrote runtime manifest to: {manifest_path}")



[DIAG] Cell 5 - Init Chroma -> FAIL: Chroma init failed. Delta embeddings will still be persisted. /Volumes/workspace/legal_data/chroma_db/: error returned from database: (code: 266) disk I/O error | /dbfs/Volumes/workspace/legal_data/chroma_db/: [Errno 5] Input/output error: '/dbfs/Volumes' | /Volumes/workspace/legal_data/chroma_db/legal_knowledge_test: error returned from database: (code: 266) disk I/O error
[12:10:54] Wrote runtime manifest to: /Volumes/workspace/legal_data/vector_db_test/embedding_runtime_manifest.json


### Diagnostics Notes
- Each major code cell writes a diagnostics status line using `report_cell(...)`.
- If Chroma still fails due environment restrictions, embeddings are safely stored in persistent Delta.
- Share the final diagnostics summary printed in Cell 9 for troubleshooting.


In [0]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)


def build_metadata(row):
    return {
        "act_name": safe_str(row.act_name),
        "section": safe_str(row.section_number),
        "category": safe_str(row.category),
        "source": safe_str(row.file_name),
    }


In [0]:
records = []
processed = 0
chroma_upserts = 0
batch_rows = []


def flush_batch(rows):
    texts = []
    ids = []
    metadatas = []

    for r in rows:
        chunk_text = safe_str(r.chunk_text).strip()
        chunk_id = safe_str(r.chunk_id).strip()

        if not chunk_text or not chunk_id:
            continue

        texts.append(chunk_text)
        ids.append(chunk_id)
        metadatas.append(build_metadata(r))

    if not ids:
        return 0, []

    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()

    ts = datetime.now(timezone.utc).isoformat()
    batch_records = []
    for i in range(len(ids)):
        batch_records.append({
            "chunk_id": ids[i],
            "chunk_text": texts[i],
            "act_name": metadatas[i]["act_name"],
            "section_number": metadatas[i]["section"],
            "category": metadatas[i]["category"],
            "file_name": metadatas[i]["source"],
            "embedding": embeddings[i],
            "embedding_model": loaded_model_name,
            "embedding_dim": len(embeddings[i]),
            "updated_at": ts,
        })

    upserted = 0
    if collection is not None:
        try:
            collection.upsert(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas,
            )
            upserted = len(ids)
        except Exception as e:
            log(f"WARNING: Chroma upsert failed for current batch. Delta export will still continue. Error: {e}")

    return upserted, batch_records


for row in gold_df.toLocalIterator():
    batch_rows.append(row)

    if len(batch_rows) >= BATCH_SIZE:
        upserted, batch_records = flush_batch(batch_rows)
        chroma_upserts += upserted
        records.extend(batch_records)
        processed += len(batch_rows)
        log(f"Processed rows: {processed}")
        batch_rows = []

if batch_rows:
    upserted, batch_records = flush_batch(batch_rows)
    chroma_upserts += upserted
    records.extend(batch_records)
    processed += len(batch_rows)

if not records:
    report_cell("Cell 8 - Build Embeddings", False, "No embedding records were produced")
    raise RuntimeError("No embedding records were produced.")

embedding_df = spark.createDataFrame(records)
embedding_df = embedding_df.withColumn("updated_at", F.to_timestamp("updated_at"))

(
    embedding_df
    .dropDuplicates(["chunk_id"])
    .write
    .format("delta")
    .mode("overwrite")
    .save(EMBEDDING_DELTA_PATH)
)

rows_written = embedding_df.count()
log(f"Delta embedding export complete at: {EMBEDDING_DELTA_PATH}")
log(f"Rows written to Delta: {rows_written}")
log(f"Vectors upserted to Chroma in this run: {chroma_upserts}")
if collection is not None:
    log(f"Current Chroma vector count: {collection.count()}")

report_cell("Cell 8 - Build Embeddings", True, f"delta_rows={rows_written}, chroma_upserts={chroma_upserts}")



[12:11:09] Processed rows: 128
[12:11:14] Processed rows: 256
[12:11:19] Processed rows: 384
[12:11:24] Processed rows: 512
[12:11:29] Processed rows: 640
[12:11:34] Processed rows: 768
[12:11:39] Processed rows: 896
[12:11:44] Processed rows: 1024
[12:11:49] Processed rows: 1152
[12:11:53] Processed rows: 1280
[12:11:58] Processed rows: 1408
[12:12:03] Processed rows: 1536
[12:12:08] Processed rows: 1664
[12:12:13] Processed rows: 1792
[12:12:18] Processed rows: 1920
[12:12:23] Processed rows: 2048
[12:12:29] Processed rows: 2176
[12:12:34] Processed rows: 2304
[12:12:40] Processed rows: 2432
[12:12:45] Processed rows: 2560
[12:12:50] Processed rows: 2688
[12:12:55] Processed rows: 2816
[12:13:00] Processed rows: 2944
[12:13:05] Processed rows: 3072
[12:13:10] Processed rows: 3200
[12:13:15] Processed rows: 3328
[12:13:19] Processed rows: 3456
[12:13:24] Processed rows: 3584
[12:13:29] Processed rows: 3712
[12:13:34] Processed rows: 3840
[12:13:38] Processed rows: 3968
[12:13:44] Proc

In [0]:
query = "What is the penalty for not wearing a helmet under Indian law?"

query_embedding = embedding_model.encode([query], show_progress_bar=False).tolist()[0]

# Smoke-test retrieval through Chroma if available
if collection is not None:
    try:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5,
        )

        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]

        print(f"Retrieved docs from Chroma: {len(docs)}")
        for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
            print(f"\n--- Chroma Result {i} ---")
            print(meta)
            print(doc[:400])

        report_cell("Cell 9 - Retrieval Smoke Test", True, f"chroma_docs={len(docs)}")

    except Exception as e:
        report_cell("Cell 9 - Retrieval Smoke Test", False, f"Chroma query failed: {e}")

else:
    # Delta fallback retrieval smoke-test
    try:
        rows = spark.read.format("delta").load(EMBEDDING_DELTA_PATH).select(
            "chunk_text", "embedding", "act_name", "section_number"
        ).limit(2000).collect()

        import math
        def cosine(a, b):
            dot = sum(x * y for x, y in zip(a, b))
            na = math.sqrt(sum(x * x for x in a)) + 1e-12
            nb = math.sqrt(sum(x * x for x in b)) + 1e-12
            return dot / (na * nb)

        scored = []
        for r in rows:
            if not r.embedding:
                continue
            s = cosine(query_embedding, [float(x) for x in r.embedding])
            scored.append((s, r.chunk_text, r.act_name, r.section_number))

        scored.sort(key=lambda x: x[0], reverse=True)
        top = scored[:3]

        print(f"Delta fallback docs: {len(top)}")
        for i, item in enumerate(top, start=1):
            print(f"\n--- Delta Result {i} ---")
            print({"score": round(item[0], 4), "act_name": item[2], "section": item[3]})
            print((item[1] or "")[:400])

        report_cell("Cell 9 - Retrieval Smoke Test", True, f"delta_docs={len(top)}, chroma_unavailable=True")

    except Exception as e:
        report_cell("Cell 9 - Retrieval Smoke Test", False, f"Delta fallback failed: {e}")

print_diagnostics_summary()



Delta fallback docs: 3

--- Delta Result 1 ---
{'score': 0.5821, 'act_name': 'Motor Vehicles Act 1988', 'section': 'Chapter V'}
-section (1) or sub-section (2) by a police officer, the owner of the vehicle shall be responsible for all towing costs, besides any other penalty. 128. Safety measures for drivers and pillion riders.—(1) No driver of a two-wheeled motor cycle shall carry more than one person in addition to himself on the motor cycle and no such person shall be carried otherwise than sitting on a proper seat secur

--- Delta Result 2 ---
{'score': 0.5196, 'act_name': 'Motor Vehicles Act 1988', 'section': 'Chapter X'}
lies a ticket of a lesser value, or (b) to check any pass or ticket, either wilfully or negligently fails or refuses to do so, he shall be punishable with fine which may extend to five hundred rupees. (3) If the holder of a permit or the driver of a contract carriage refuses, in contravention of the provisions of this Act or rules made thereunder, to ply the contr